# 01 — Data Exploration: MIT-BIH Arrhythmia Database

This notebook provides interactive exploration of the MIT-BIH ECG data:
1. Load and visualize raw ECG signals
2. Inspect R-peak annotations
3. Examine extracted beat windows
4. Visualize CWT scalograms
5. Dataset statistics and class distribution

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import wfdb
import pywt
from pathlib import Path

from src.config import (
    DATA_RAW, DATA_PROCESSED, DATA_SCALOGRAMS,
    SAMPLING_RATE, WINDOW_BEFORE, WINDOW_AFTER, BEAT_LENGTH,
    NORMAL_LABELS, ABNORMAL_LABELS, CLASS_NAMES,
    CWT_WAVELET, CWT_SCALES_MAX,
)

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 4)
plt.rcParams['figure.dpi'] = 120

## 1. Load a Raw ECG Record

We'll use **Record 100** as our example — a common benchmark record from the MIT-BIH database.

In [ ]:
RECORD_ID = '100'
record_path = str(DATA_RAW / RECORD_ID)

# Read signal and annotations
record = wfdb.rdrecord(record_path, channels=[0])
annotation = wfdb.rdann(record_path, 'atr')

signal = record.p_signal.flatten()
print(f'Record {RECORD_ID}')
print(f'  Signal length : {len(signal):,} samples')
print(f'  Duration      : {len(signal)/SAMPLING_RATE:.1f} seconds ({len(signal)/SAMPLING_RATE/60:.1f} min)')
print(f'  Sampling rate : {SAMPLING_RATE} Hz')
print(f'  Signal range  : [{signal.min():.4f}, {signal.max():.4f}]')
print(f'  Annotations   : {len(annotation.sample)} beats')

### 1.1 Plot the Full ECG Signal (first 10 seconds)

In [ ]:
# Plot first 10 seconds
t_end = 10  # seconds
n_samples = int(t_end * SAMPLING_RATE)
time_axis = np.arange(n_samples) / SAMPLING_RATE

fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(time_axis, signal[:n_samples], color='#2c3e50', linewidth=0.8)

# Mark R-peaks
rpeak_mask = annotation.sample < n_samples
rpeak_samples = annotation.sample[rpeak_mask]
rpeak_times = rpeak_samples / SAMPLING_RATE
ax.scatter(rpeak_times, signal[rpeak_samples], color='#e74c3c', s=40, zorder=5, label='R-peaks')

# Annotate beat types
for s, sym in zip(annotation.sample[rpeak_mask], np.array(annotation.symbol)[rpeak_mask]):
    ax.annotate(sym, (s/SAMPLING_RATE, signal[s]), fontsize=7,
                textcoords='offset points', xytext=(0, 12), ha='center', color='#e74c3c')

ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude (mV)')
ax.set_title(f'ECG Signal — Record {RECORD_ID} (MLII lead, first {t_end}s)')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

## 2. Annotation Distribution

Let's look at the distribution of beat types in this record and across the full database.

In [ ]:
from collections import Counter

# Single record
symbol_counts = Counter(annotation.symbol)
print(f'Beat-type distribution for Record {RECORD_ID}:')
for sym, count in symbol_counts.most_common():
    label_type = 'Normal' if sym in NORMAL_LABELS else ('Abnormal' if sym in ABNORMAL_LABELS else 'Non-beat')
    print(f"  '{sym}' : {count:5d}  ({label_type})")

In [ ]:
# Plot annotation histogram
symbols = list(symbol_counts.keys())
counts = list(symbol_counts.values())

colors = ['#27ae60' if s in NORMAL_LABELS else '#e74c3c' if s in ABNORMAL_LABELS else '#95a5a6' for s in symbols]

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(symbols, counts, color=colors, edgecolor='white')
ax.set_xlabel('Beat Type Symbol')
ax.set_ylabel('Count')
ax.set_title(f'Annotation Distribution — Record {RECORD_ID}')

# Custom legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#27ae60', label='Normal'),
    Patch(facecolor='#e74c3c', label='Abnormal'),
    Patch(facecolor='#95a5a6', label='Non-beat'),
]
ax.legend(handles=legend_elements)
plt.tight_layout()
plt.show()

## 3. Extracted Beat Windows

Visualize individual heartbeat segments extracted around the R-peak.

In [ ]:
# Extract a few example beats from this record
example_beats = []
example_labels = []

for sample, symbol in zip(annotation.sample, annotation.symbol):
    start = sample - WINDOW_BEFORE
    end = sample + WINDOW_AFTER
    if start < 0 or end > len(signal):
        continue
    if symbol in NORMAL_LABELS:
        example_beats.append(signal[start:end])
        example_labels.append('Normal')
    elif symbol in ABNORMAL_LABELS:
        example_beats.append(signal[start:end])
        example_labels.append('Abnormal')
    if len(example_beats) >= 200:
        break

print(f'Extracted {len(example_beats)} beats from Record {RECORD_ID}')

In [ ]:
# Plot 5 Normal + 5 Abnormal beats side by side
normal_beats = [b for b, l in zip(example_beats, example_labels) if l == 'Normal'][:5]
abnormal_beats = [b for b, l in zip(example_beats, example_labels) if l == 'Abnormal'][:5]

fig, axes = plt.subplots(2, 5, figsize=(18, 6), sharey=True)
time_beat = np.arange(BEAT_LENGTH) / SAMPLING_RATE * 1000  # ms

for i, beat in enumerate(normal_beats):
    axes[0, i].plot(time_beat, beat, color='#27ae60', linewidth=1)
    axes[0, i].set_title(f'Normal #{i+1}', fontsize=10)
    axes[0, i].axvline(x=WINDOW_BEFORE/SAMPLING_RATE*1000, color='red', linestyle='--', alpha=0.5, linewidth=0.8)
    if i == 0:
        axes[0, i].set_ylabel('Amplitude (mV)')

for i, beat in enumerate(abnormal_beats[:5]):
    axes[1, i].plot(time_beat, beat, color='#e74c3c', linewidth=1)
    axes[1, i].set_title(f'Abnormal #{i+1}', fontsize=10)
    axes[1, i].axvline(x=WINDOW_BEFORE/SAMPLING_RATE*1000, color='red', linestyle='--', alpha=0.5, linewidth=0.8)
    axes[1, i].set_xlabel('Time (ms)')
    if i == 0:
        axes[1, i].set_ylabel('Amplitude (mV)')

# Hide empty subplots if fewer than 5 abnormal
for i in range(len(abnormal_beats), 5):
    axes[1, i].set_visible(False)

fig.suptitle(f'Individual Heartbeat Windows — Record {RECORD_ID}', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 4. CWT Scalogram Visualization

Demonstrate the Continuous Wavelet Transform applied to individual beats.

In [ ]:
scales = np.arange(1, CWT_SCALES_MAX)

def plot_beat_and_scalogram(beat, title='Beat'):
    """Plot a beat and its CWT scalogram side by side."""
    coefficients, frequencies = pywt.cwt(beat, scales, CWT_WAVELET,
                                         sampling_period=1.0/SAMPLING_RATE)
    scalogram = np.abs(coefficients)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    
    # 1D signal
    t = np.arange(len(beat)) / SAMPLING_RATE * 1000
    ax1.plot(t, beat, color='#2c3e50', linewidth=1)
    ax1.set_xlabel('Time (ms)')
    ax1.set_ylabel('Amplitude')
    ax1.set_title(f'{title} — 1D Signal')
    ax1.axvline(x=WINDOW_BEFORE/SAMPLING_RATE*1000, color='red', linestyle='--', alpha=0.5)
    
    # Scalogram
    im = ax2.imshow(scalogram, aspect='auto', cmap='jet',
                    extent=[0, len(beat)/SAMPLING_RATE*1000, frequencies[-1], frequencies[0]])
    ax2.set_xlabel('Time (ms)')
    ax2.set_ylabel('Frequency (Hz)')
    ax2.set_title(f'{title} — CWT Scalogram')
    plt.colorbar(im, ax=ax2, label='Magnitude')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Plot a Normal beat and its scalogram
if normal_beats:
    plot_beat_and_scalogram(normal_beats[0], title='Normal Beat')

# Plot an Abnormal beat and its scalogram
if abnormal_beats:
    plot_beat_and_scalogram(abnormal_beats[0], title='Abnormal Beat')

## 5. Processed Dataset Statistics

If you have already run the extraction pipeline (`python -m src.extract_beats`), we can inspect the saved data.

In [ ]:
beats_path = DATA_PROCESSED / 'beats.npy'
labels_path = DATA_PROCESSED / 'labels.npy'

if beats_path.exists() and labels_path.exists():
    beats = np.load(beats_path)
    labels = np.load(labels_path)
    
    print(f'Processed dataset:')
    print(f'  beats.npy  shape: {beats.shape}  dtype: {beats.dtype}')
    print(f'  labels.npy shape: {labels.shape}  dtype: {labels.dtype}')
    print(f'  Normal     : {(labels == 0).sum():,}')
    print(f'  Abnormal   : {(labels == 1).sum():,}')
    print(f'  Total      : {len(labels):,}')
    
    # Class distribution pie chart
    fig, ax = plt.subplots(figsize=(5, 5))
    counts = [(labels == 0).sum(), (labels == 1).sum()]
    ax.pie(counts, labels=CLASS_NAMES, autopct='%1.1f%%',
           colors=['#27ae60', '#e74c3c'], startangle=90,
           textprops={'fontsize': 12})
    ax.set_title('Class Distribution (after balancing)')
    plt.show()
else:
    print('Processed data not found. Run: python -m src.extract_beats')

## 6. Scalogram Gallery

Preview saved scalogram images from the dataset.

In [ ]:
from PIL import Image

for cls_name in CLASS_NAMES:
    cls_dir = DATA_SCALOGRAMS / cls_name
    images = sorted(cls_dir.glob('*.png'))[:5]
    
    if not images:
        print(f'No scalograms found in {cls_dir}. Run: python -m src.build_scalograms')
        continue
    
    fig, axes = plt.subplots(1, len(images), figsize=(15, 3))
    if len(images) == 1:
        axes = [axes]
    
    for i, img_path in enumerate(images):
        img = Image.open(img_path)
        axes[i].imshow(img)
        axes[i].set_title(img_path.stem, fontsize=8)
        axes[i].axis('off')
    
    fig.suptitle(f'{cls_name} Scalograms', fontsize=13)
    plt.tight_layout()
    plt.show()

---

## Next Steps

After exploring the data, proceed with the training pipeline:

```bash
# Train SmallNet
python -m src.train --model smallnet --epochs 50

# Train GoogLeNet
python -m src.train --model googlenet --epochs 50

# Evaluate & compare
python -m src.evaluate --model smallnet
python -m src.evaluate --model googlenet
python -m src.evaluate --compare
```